# 04 · DIRE pipeline — tổng quan, trạng thái và bước chạy

Notebook này **trình bày** pipeline và snapshot inventory do người vận hành cung cấp; các ô không khởi chạy GPU hay sửa Volume.

> **Về Modal:** Modal đã ngừng trình soạn thảo Modal Notebooks. Đây là file `.ipynb` tiêu chuẩn, có thể mở bằng JupyterLab; nếu cần Jupyter chạy trên Modal thì tự host Jupyter server qua Modal Tunnel. [Modal Notebooks](modal.com/docs/guide/notebooks.md) · [Jupyter qua Tunnel](modal.com/docs/guide/tunnels).


## Luồng chính

```text
ảnh + manifest
    │  kiểm tra split / generator / SHA; ghi nhận dữ liệu lỗi
    ▼
ADM diffusion checkpoint (cố định)
    │  inversion: S lượt mạng
    ▼
trạng thái nhiễu x_T
    │  reconstruction: S lượt mạng
    ▼
ảnh tái tạo R(I(x)) ──────┐
                           ├─ DIRE map = |x − R(I(x))|
ảnh đầu vào x ────────────┘
    │  shape dự kiến (3, 256, 256), float16
    ▼
ResNet-50 detector (mỗi generator một detector)
    │  threshold chọn trên validation
    ▼
đánh giá test / báo cáo / so sánh
```

Với `S=20`, một map cần `2S=40` NFE. Map là sai khác theo pixel, không phải nhãn dự đoán; ResNet-50 mới là bộ phân loại.


## Snapshot inventory hiện tại

Bảng sau là snapshot người vận hành gửi, không tự đọc Modal Volume. `ảnh` là số ảnh trên Volume, `cần` là số được chọn theo `--n`, `map` là map hiện có.


In [ ]:
cells = [
    ("test/ddpm", 768, 768, 768, 0),
    ("test/midjourney", 100, 100, 100, 0),
    ("test/real", 1000, 1000, 1000, 0),
    ("test/stylegan", 1000, 1000, 1000, 0),
    ("train/real", 40000, 1000, 1000, 0),
    ("train/stylegan", 34124, 1000, 1000, 0),
    ("val/real", 1000, 1000, 1000, 0),
    ("val/stylegan", 1000, 1000, 1000, 0),
]
headers = ("Ô", "Ảnh", "Cần", "Map", "Thiếu")
widths = [max(len(headers[i]), max(len(str(row[i])) for row in cells)) for i in range(len(headers))]
print(" | ".join(headers[i].ljust(widths[i]) for i in range(len(headers))))
print("-+-".join("-" * w for w in widths))
for row in cells:
    print(" | ".join(str(row[i]).ljust(widths[i]) for i in range(len(row))))
print(f"\nTổng map: {sum(r[3] for r in cells):,}; thiếu: {sum(r[4] for r in cells):,}")
assert sum(r[3] for r in cells) == 6868
assert sum(r[4] for r in cells) == 0


## Ý nghĩa cho training

- Có `train/real` + `train/stylegan`, và `val/real` + `val/stylegan`: hiện có dữ liệu cho detector StyleGAN.
- DDPM và Midjourney chỉ ở test trong snapshot này; giữ chúng làm holdout, không train trên chúng.
- `test/real=1000` và `test/midjourney=100` tạo tỷ lệ `10:1`; đọc recall Midjourney cùng số mẫu và metric khác.
- “Thiếu 0” xác nhận inventory, **không tự chứng minh mọi `.npy` đều đọc được**. `maps --verify` đọc thử map.


In [ ]:
real_test = next(row[3] for row in cells if row[0] == "test/real")
mj_test = next(row[3] for row in cells if row[0] == "test/midjourney")
print(f"Test real : Midjourney = {real_test}:{mj_test} = {real_test / mj_test:.1f}:1")


## Các lệnh nối pipeline

Các lệnh dưới đây chỉ hiển thị để copy; notebook không tự gọi Modal và không phát sinh chi phí.


In [ ]:
RUN_ID = "rebuild-2026-10-14"
commands = [
    "modal run -m dire_zero.modal_app::checkdata",
    f"modal run -m dire_zero.modal_app::maps --n 1000 --steps 20 --generator ddpm,midjourney,stylegan --run-id {RUN_ID} --verify 20",
    f"modal run -m dire_zero.modal_app::train --run-id {RUN_ID} --generator stylegan --epochs 40",
    f"modal run -m dire_zero.modal_app::evaluate --run-id {RUN_ID} --generator stylegan --split test",
    f"modal run -m dire_zero.modal_app::report --run-id {RUN_ID}",
]
print("\n".join(commands))


## Phạm vi và diễn giải

`train --epochs 40` hiện là lượt thăm dò, khởi tạo scratch; chưa xác nhận là protocol bài báo. `evaluate` hiện chấm generator đã train, chưa tạo ma trận cross-generator đầy đủ cho DDPM/Midjourney. Không gọi đây là tái hiện paper chính xác nếu chưa chốt protocol.

Nếu đã xóa ảnh 0 byte, đối chiếu `manifest.tsv` từ report xóa rồi chạy `checkdata`. Map rebuild nằm ở `/zero/rebuild/maps/<run_id>`; train/eval/report dùng cùng `--run-id`, không chép đè `/zero/maps`.
